# Consignors Dimension — Gold Layer (`company_db.gold.consignor_dim`)

This notebook builds and maintains the **Consignors Dimension** table in the Gold layer of the Company data warehouse. It is the canonical reference for every consignor who has submitted lots to an Company auction, enriched with contact details and classification attributes from the Bronze consignors register.

---

## What This Notebook Does

1. Extracts distinct consignor records from Silver auction results, enriched with profile data from the Bronze consignors table
2. Controls whether this is an initial or incremental load via the `flag` widget
3. Initialises the `consignor_dim` Delta table if it does not already exist
4. Assigns or preserves surrogate keys (`consignor_dim_key`)
5. Merges the result into the Gold dimension table (insert new consignors, update existing ones)

---

## Source Tables

| Table | Layer | Description |
|---|---|---|
| `company_db.silver.auctionresults` | Silver | Driving table — all consignors who have appeared in auction results |
| `company_db.bronze.consignors` | Bronze | Consignor profile register with classification and location attributes (RIGHT JOIN — consignors not in Bronze still appear with NULL `dcc` / `province`) |

## Target Table

`company_db.gold.consignor_dim`  
Stored in Azure Data Lake Storage Gen2: `abfss://auction@company.dfs.core.windows.net/gold/consignor_dim`

---

## MERGE Key

Records are matched on the **single natural key**: `consignor_id`  
One row per consignor across all auctions.

---

## Load Strategy (controlled by the `flag` widget)

| `flag` value | Behavior |
|---|---|
| `0` | **Initial / Full Load** — surrogate key sequence starts at 1 |
| `1` | **Incremental Load** — surrogate key sequence continues from current `MAX(consignor_dim_key)` |

---

## Column Reference

| Column | Type | Description |
|---|---|---|
| `consignor_dim_key` | BIGINT | **Surrogate key** — auto-incrementing warehouse identifier |
| `consignor_id` | DOUBLE | Source system consignor identifier (natural / business key) |
| `consignor_name` | STRING | Consignor’s full name or business name |
| `consignor_email` | STRING | Consignor’s email address (join key to Bronze consignors table) |
| `dcc` | STRING | Dealer / Consignor Classification code (from Bronze; NULL if not registered) |
| `province` | STRING | Province or state of the consignor (from Bronze; NULL if not registered) |
| `created_at` | TIMESTAMP | Timestamp when the row was first inserted |
| `updated_at` | TIMESTAMP | Timestamp of the last MERGE update |

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

---
## Step 1 · Extract Consignors from Bronze & Silver

Builds the distinct consignor dataset by joining two tables:

| Join | Type | Key | Purpose |
|---|---|---|---|
| `silver.auctionresults` ← `bronze.consignors` | RIGHT JOIN on `email = consignor_email` | `consignor_email` | Enrich auction consignors with profile data |

The **RIGHT JOIN** keeps `silver.auctionresults` as the driving table — every consignor who has appeared in at least one auction result is included, even if they have no record in the Bronze register. In that case, `dcc` and `province` will be `NULL`.

`DISTINCT(consignor_id)` ensures one row per consignor regardless of how many lots they have submitted.

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
consignor_table = spark.sql(f"""
SELECT 
DISTINCT(ar.consignor_id),
ar.consignor_name,
ar.consignor_email,
c.dcc,
c.province

FROM
{catalog}.bronze.consignors as c
right JOIN {catalog}.silver.auctionresults as ar
ON c.email = ar.consignor_email

"""
)


---
## Step 2 · Load Control Flag

The `flag` widget controls the load mode for this run:
- `flag = 0` — **Initial / Full Load.** Surrogate key counter (`consignor_dim_key`) starts at 1. Use this on the first ever run or after a full table reset.
- `flag = 1` — **Incremental Load.** Surrogate key counter continues from `MAX(consignor_dim_key)` in the existing table. Use this for every scheduled run.

In [ ]:
# flag =  dbutils.widgets.text('flag','0')
# flag = dbutils.widgets.get('flag')
# flag

---
## Step 3 · Initialize Gold Dimension Table

Creates `company_db.gold.consignor_dim` as a **Delta table** if it does not already exist. On subsequent runs this is a no-op — no existing data is affected.

Granularity: **one row per `consignor_id`** (enforced by the MERGE key in Step 6).

The table is backed by ADLS Gen2 at `gold/consignor_dim` and supports full ACID transactions (MERGE, time travel, schema evolution).

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.consignor_dim
(
  consignor_dim_key BIGINT,
  consignor_id DOUBLE,
  consignor_name STRING,
  consignor_email STRING,
  dcc STRING,
  province STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/gold/consignor_dim';""")

In [ ]:
consignor_dim = spark.table(f'{catalog}.gold.consignor_dim')

---
## Step 4 · Identify New vs Existing Consignors

Loads the current `company_db.gold.consignor_dim` and LEFT JOINs it against the incoming consignors on `consignor_id`.

After the join, records are split into two groups:

| Group | Condition | Next Action |
|---|---|---|
| `new_consignors` | `consignor_dim_key IS NULL` | Needs a surrogate key assigned, then INSERTed |
| `old_consignors` | `consignor_dim_key IS NOT NULL` | Surrogate key already exists, will be UPDATEd via MERGE |

In [ ]:
consignor_table = consignor_table.join(
    consignor_dim,
    consignor_dim.consignor_id == consignor_table.consignor_id,
    'left'
).select(
    consignor_table.consignor_id,
    consignor_table.consignor_name,
    consignor_table.consignor_email,
    consignor_table.dcc,
    consignor_table.province,
    consignor_dim.consignor_dim_key
).sort('consignor_id')

In [ ]:
consignor_table= consignor_table.dropna(subset=['consignor_id'])
consignor_table = consignor_table.dropDuplicates(subset=['consignor_id'])

In [ ]:
new_consignors = consignor_table.filter(consignor_table.consignor_dim_key.isNull())
old_consignors= consignor_table.filter(consignor_table.consignor_dim_key.isNotNull())

---
## Step 5 · Surrogate Key Assignment for New Consignors

Assigns a unique `consignor_dim_key` to each new consignor using `ROW_NUMBER()` ordered by `consignor_id` (lowest source ID gets the lowest key).

The key is offset by `max_value` to avoid collisions with existing records:
```
consignor_dim_key = ROW_NUMBER() OVER (ORDER BY consignor_id) + max_value
```

- On an **initial load** (`flag=0`): `max_value = 0`, so keys start at 1.
- On an **incremental load** (`flag=1`): `max_value = MAX(consignor_dim_key)` from the existing table, so keys continue seamlessly.

In [ ]:
#  #fetching max_value from existing table
# if flag == '0':
#     max_value = 0
# else:
#     max_value = spark.sql(""" SELECT MAX(consignor_dim_key) FROM company_db.gold.consignor_dim """).collect()[0][0]


In [ ]:
max_value = spark.sql(f"""
    SELECT COALESCE(MAX(lot_dim_key), 0) FROM {catalog}.gold.lots_dim
""").collect()[0][0]

In [ ]:
# new_users = new_users.withColumn("user_dim_key", ((F.monotonically_increasing_id()+1)+max_value)).display()
from pyspark.sql import Window
window_spec = Window.orderBy("consignor_id")

new_consignors = new_consignors.withColumn(
    "consignor_dim_key",
    (F.row_number().over(window_spec) + F.lit(max_value))
)

In [ ]:
final_consignors = old_consignors.union(new_consignors)

---
## Step 6 · MERGE into Gold Layer

`old_consignors` (existing, with keys) and `new_consignors` (newly keyed) are unioned into `final_consignors` and registered as a Spark temp view.

The `MERGE INTO company_db.gold.consignor_dim` applies two operations atomically on the single key `consignor_id`:

| Condition | Action | Notes |
|---|---|---|
| MATCHED | **UPDATE** name, email, dcc, province + `updated_at = current_timestamp()` | Keeps consignor contact and classification details in sync |
| NOT MATCHED | **INSERT** full row with `created_at = current_timestamp()`, `updated_at = current_timestamp()` | Adds new consignors |

> `consignor_dim_key` is only written on **INSERT** — not updated on MATCH — to preserve the stable surrogate key.

In [ ]:
final_consignors.createOrReplaceTempView("final_consignors")

In [ ]:
spark.sql(f"""
MERGE INTO {catalog}.gold.consignor_dim AS target
USING final_consignors AS source
ON target.consignor_id = source.consignor_id
WHEN MATCHED THEN
UPDATE SET
  target.consignor_name = source.consignor_name,
  target.consignor_email = source.consignor_email,
  target.dcc = source.dcc,
  target.province = source.province,
  target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (
  consignor_dim_key, consignor_id, consignor_name, consignor_email,
  dcc, province, created_at, updated_at
)
VALUES (
  source.consignor_dim_key, source.consignor_id, source.consignor_name,
  source.consignor_email, source.dcc, source.province,
  current_timestamp(), current_timestamp()
)""")

---
## Step 7 · Verification

Queries the final `company_db.gold.consignor_dim` table to validate the MERGE output. Spot-check that:
- `consignor_dim_key` values are sequential with no gaps for new consignors
- `created_at` and `updated_at` are populated for all rows
- Consignors not in the Bronze register (NULL `dcc` / `province`) are expected — verify the count is reasonable
- No duplicate `consignor_id` values exist

In [ ]:

# Verify MERGE output: row counts, key range, and Bronze enrichment coverage
spark.sql(f"""
SELECT
  COUNT(*)                                                  AS total_consignors,
  MIN(consignor_dim_key)                                    AS min_dim_key,
  MAX(consignor_dim_key)                                    AS max_dim_key,
  COUNT(DISTINCT consignor_id)                              AS unique_consignor_ids,
  SUM(CASE WHEN dcc IS NULL THEN 1 END)                     AS missing_dcc,
  SUM(CASE WHEN province IS NULL THEN 1 END)                AS missing_province,
  SUM(CASE WHEN consignor_email IS NULL THEN 1 END)         AS missing_email,
  MAX(updated_at)                                           AS latest_updated
FROM {catalog}.gold.consignor_dim;""")

In [ ]:
# %sql
# DESCRIBE HISTORY company_db.gold.consignor_dim;
# RESTORE TABLE company_db.gold.consignor_dim TO VERSION AS OF 5;